# Reranker comparison on UCI Online Retail II

Is LightGBM LambdaRank the right Stage 1 reranker? This notebook trains five alternatives on exactly the same inputs and scores them on the same test baskets:

* **Same candidates:** the 150-item pool from the 11 retrieval sources (`configs/default.yaml`).
* **Same 35 features**, the same training events (validation invoices, 3 hidden items each, candidates from train-only models) and the same early-stopping slice.
* **Same test baskets:** 1,000 UCI test invoices sampled with seed 42.
* CPU only (4-core cloud container). Confidence intervals: 2,000 bootstrap resamples over test baskets, paired against LightGBM.

The code lives in `scripts/reranker_comparison.py`; this notebook calls it step by step so every output below is from a real run.

In [ ]:
import sys, time, platform, os
sys.path.insert(0, '../scripts')
import yaml, numpy as np, pandas as pd
import lightgbm, xgboost, catboost, sklearn
import reranker_comparison as rc

os.chdir('..')  # repository root, so configs/ and data/ resolve
cfg = yaml.safe_load(open('configs/default.yaml'))
print(platform.python_version(), platform.machine(), os.cpu_count(), 'CPUs')
print('lightgbm', lightgbm.__version__, '| xgboost', xgboost.__version__, '| catboost', catboost.__version__, '| sklearn', sklearn.__version__)
print('seed', cfg['seed'], '| pool size', cfg['retrieval']['pool_size'])

## 1. Candidate pools and features

This is the slow step (about 5 minutes): every retrieval source scores every event.

In [ ]:
t0 = time.perf_counter()
feats_val, feats_test = rc.build_features(cfg, n_test=1000)
print(f'built in {time.perf_counter() - t0:.0f} s')
cols = rc.feature_columns(feats_val)
print('features:', len(cols))
print('training rows:', len(feats_val), '| test rows:', len(feats_test), '| test events:', feats_test.event_id.nunique())
in_pool = feats_test.groupby('event_id').label.max()
print(f'pool recall on this sample: {in_pool.mean():.1%} (ceiling for every model)')

## 2. Train and score the six rerankers

In [ ]:
summary, ranks = rc.compare(cfg, feats_val, feats_test, n_boot=cfg['stats']['n_boot'])
out = 'results/reranker_comparison'
os.makedirs(out, exist_ok=True)
summary.to_csv(f'{out}/summary.csv', index=False)
ranks.to_csv(f'{out}/ranks.csv', index=False)

## 3. Results

In [ ]:
t = summary.copy()
fmt = lambda x: '' if pd.isna(x) else f'{x:+.1f}'
show = pd.DataFrame({
    'Model': t['model'],
    'Hit@10 %': (100 * t['hit@10']).round(1),
    'Hit@10 95% CI': [f'[{100*a:.1f}, {100*b:.1f}]' for a, b in zip(t['hit@10_ci_low'], t['hit@10_ci_high'])],
    'Hit@2 %': (100 * t['hit@2']).round(1),
    'NDCG@10': t['ndcg@10'].round(3),
    'Train s': t['train_s'].round(0),
    'Score s (1,000 baskets)': t['score_s'].round(2),
    'Δ Hit@10 vs LightGBM': [fmt(100*x) for x in t.get('d_hit@10')],
    'Δ 95% CI': ['' if pd.isna(a) else f'[{100*a:+.1f}, {100*b:+.1f}]' for a, b in zip(t['d_ci_low'], t['d_ci_high'])],
})
show

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.barh(t['model'][::-1], 100 * t['hit@10'][::-1], xerr=[100*(t['hit@10']-t['hit@10_ci_low'])[::-1], 100*(t['hit@10_ci_high']-t['hit@10'])[::-1]], color='#4C72B0', capsize=3)
for y, (h, s) in enumerate(zip(100 * t['hit@10'][::-1], t['train_s'][::-1])):
    ax.text(1, y, f'{h:.1f}%  ·  train {s:.0f} s', va='center', color='white', fontsize=8)
ax.axvline(100 * t['pool_recall'].iloc[0], ls='--', color='grey'); ax.set_xlabel('Hit@10 on 1,000 test baskets (%)')
plt.tight_layout(); plt.show()

## Reading

* A model ties LightGBM when its paired Δ interval contains zero; it loses when the whole interval is below zero.
* Every model is capped by pool recall (dashed line): a basket whose hidden item is not in the 150 candidates is a miss for all of them.
* On this 1,000-basket sample LightGBM reaches Hit@10 38.6% (pool recall 65.5%), close to the full 4,480-basket test set (38.1%, pool recall 65.3%). Compare models within this table.